In [ ]:
# 03_train_explainable_models.ipynb
# Explainable ML – Software Defect Prediction
# Decision Tree, Logistic Regression, Naive Bayes

import sys
sys.path.append("../src")
from utils import DATASETS, TARGET, RANDOM_STATE, N_SPLITS

import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import copy

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

import matplotlib.pyplot as plt
import seaborn as sns

processed_path = Path("../datasets/processed")
results_path = Path("../results/explainable_models")
results_path.mkdir(parents=True, exist_ok=True)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def evaluate_fold(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

models = {
    "Decision_Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),

    "Logistic_Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=500
        ))
    ]),

    "Naive_Bayes": GaussianNB()
}

results = []

for ds in DATASETS:
    print(f" DATASET: {ds}")

    dataset_path = results_path / ds
    dataset_path.mkdir(parents=True, exist_ok=True)

    data = pd.read_csv(processed_path / f"{ds}_processed.csv")
    X = data.drop(columns=[TARGET])
    y = data[TARGET]

    for model_name, model in models.items():
        print(f" → {model_name}")

        model_path = dataset_path / model_name
        model_path.mkdir(parents=True, exist_ok=True)

        fold_metrics = []
        best_model = copy.deepcopy(model)  
        best_f1 = -1

        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            smote = SMOTE(random_state=RANDOM_STATE)
            resampled = smote.fit_resample(X_train, y_train)
            X_train_res, y_train_res = resampled[0], resampled[1]

            fold_model = copy.deepcopy(model)
            fold_model.fit(X_train_res, y_train_res)
            y_pred = fold_model.predict(X_test)

            fm = evaluate_fold(y_test, y_pred)
            fold_metrics.append(fm)

            if fm["F1-score"] > best_f1:
                best_f1 = fm["F1-score"]
                best_model = fold_model

        avg_metrics: dict = {
            "Dataset": ds,
            "Model": model_name,
            "Accuracy": round(float(np.mean([m["Accuracy"] for m in fold_metrics])), 4),
            "Precision": round(float(np.mean([m["Precision"] for m in fold_metrics])), 4),
            "Recall": round(float(np.mean([m["Recall"] for m in fold_metrics])), 4),
            "F1-score": round(float(np.mean([m["F1-score"] for m in fold_metrics])), 4)
        }
        results.append(avg_metrics)

        joblib.dump(best_model, model_path / "model.joblib")

        if model_name == "Decision_Tree":

            tree_full = DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                min_samples_leaf=30
            )

            best_fold_idx = np.argmax([m["F1-score"] for m in fold_metrics])
            train_idx_best = list(cv.split(X, y))[best_fold_idx][0]
            X_tr = X.iloc[train_idx_best]
            y_tr = y.iloc[train_idx_best]
            smote_vis = SMOTE(random_state=RANDOM_STATE)
            resampled_vis = smote_vis.fit_resample(X_tr, y_tr)
            X_tr_res, y_tr_res = resampled_vis[0], resampled_vis[1]

            tree_full.fit(X_tr_res, y_tr_res)
            joblib.dump(tree_full, model_path / "model_full.joblib")

            plt.figure(figsize=(22, 10))
            plot_tree(
                tree_full,
                feature_names=X.columns,
                class_names=["No Defect", "Defect"],
                filled=True,
                fontsize=10
            )
            plt.title(f"{ds} – Full Tree")
            plt.tight_layout()
            plt.savefig(model_path / "tree_full.png", dpi=300)
            plt.close()

            tree_pruned = DecisionTreeClassifier(
                random_state=RANDOM_STATE,
                max_depth=4,
                min_samples_leaf=30
            )
            tree_pruned.fit(X_tr_res, y_tr_res)
            joblib.dump(tree_pruned, model_path / "model_pruned.joblib")

            plt.figure(figsize=(18, 8))
            plot_tree(
                tree_pruned,
                feature_names=X.columns,
                class_names=["No Defect", "Defect"],
                filled=True,
                fontsize=12
            )
            plt.title(f"{ds} – Pruned Tree")
            plt.tight_layout()
            plt.savefig(model_path / "tree_pruned.png", dpi=300)
            plt.close()

            importance = pd.DataFrame({
                "Feature": X.columns,
                "Importance": tree_full.feature_importances_
            }).sort_values(by="Importance", ascending=False)

            importance.to_csv(model_path / "feature_importance.csv", index=False)

            plt.figure(figsize=(8, 6))
            sns.barplot(
                data=importance,
                x="Importance",
                y="Feature",
                hue="Feature",
                palette="coolwarm",
                legend=False
            )
            plt.title("Feature Importance - Full Tree")
            plt.tight_layout()
            plt.savefig(model_path / "feature_importance.png", dpi=300)
            plt.close()

        if model_name == "Logistic_Regression":

            logreg = best_model.named_steps["logreg"]

            coef_df = pd.DataFrame({
                "Feature": X.columns,
                "Coefficient": logreg.coef_[0]
            }).sort_values(by="Coefficient")

            coef_df.to_csv(model_path / "coefficients.csv", index=False)

            plt.figure(figsize=(8, 6))
            sns.barplot(
                x="Coefficient",
                y="Feature",
                hue="Feature",
                data=coef_df,
                palette="viridis",
                legend=False
            )
            plt.title(f"Logistic Regression Coefficients - {ds}", fontsize=14, weight='bold')
            plt.xlabel("Coefficient value", fontsize=12)
            plt.ylabel("Feature", fontsize=12)
            plt.tight_layout()
            plt.savefig(results_path / f"{ds}_logistic_regression_coefficients.png", dpi=300)
            plt.close()

        if model_name == "Naive_Bayes":

            means = pd.DataFrame(
                best_model.theta_,
                columns=X.columns,
                index=["No Defect", "Defect"]
            ).T

            means.to_csv(model_path / "means.csv")

            plt.figure(figsize=(10, 6))
            sns.heatmap(means, annot=True, fmt=".2f", cmap="magma",
                        linewidths=0.5, linecolor='white', annot_kws={"size": 12})
            plt.title(f"Class-conditional means - Naive Bayes ({ds})", fontsize=14, weight='bold')
            plt.xlabel("Clase", fontsize=12)
            plt.ylabel("Feature", fontsize=12)
            plt.xticks(rotation=45, ha="right", fontsize=10)
            plt.yticks(fontsize=12)
            plt.tight_layout()
            plt.savefig(results_path / f"{ds}_naive_bayes_means.png", dpi=300)
            plt.close()

results_df = pd.DataFrame(results)
results_df = results_df[
    ["Dataset", "Model", "Accuracy", "Precision", "Recall", "F1-score"]
]

results_df.to_csv(
    results_path / "explainable_models_metrics.csv",
    index=False
)

results_df
